# Nifty + Bank Nifty Options – Paper Trading (23 Oct 2026)

**सिर्फ़ PAPER TRADING.** इस notebook में ऑर्डर लगाने वाला कोई code नहीं है। यह Dhan से सिर्फ़ market data पढ़ता है, rules के हिसाब से trades simulate करता है और P&L निकालता है।

यह वही process है जो 23 Oct 2026 को चला था (method: day-trader skill का *trader-entry / trader-exit / trader-budget / trader-risk*):

| Rule | Value |
|---|---|
| Instruments | NIFTY और BANKNIFTY index options, expiry 27 Oct 2026, सिर्फ़ option **buying** |
| Opening range | 9:15–9:30 का index high / low |
| Entry | 9:30 के बाद पहली 5-min index candle range high के **ऊपर close** → ATM CE buy; range low के **नीचे close** → ATM PE buy (entry: उसी 5-min candle का option close + slippage) |
| Exit | Stop = premium −30% · Target = 2R (~+60%) · flat by 15:20 · एक candle में stop और target दोनों छुए तो stop माना जाता है |
| Risk | Paper capital ₹5,00,000 · max loss per trade ₹5,000 · हर index पर max 2 trades (दूसरा सिर्फ़ stop-out के बाद, उल्टी तरफ़) · combined daily loss limit ₹15,000 |
| Charges | brokerage ₹20/order, STT 0.15% (sell), exchange 0.03503%, SEBI, stamp 0.003% (buy), GST 18% |

**कैसे चलाएँ:** `Runtime → Run all`। Default `MODE = "replay"` बिना login के 23 Oct का पूरा दिन दोबारा चलाता है और वही नतीजा देता है।
नए दिन के लिए `MODE = "live"` रखें: Dhan **client-id** और **access token** पूछा जाएगा (Dhan web → My Profile → DhanHQ Trading APIs)। Token कभी notebook में लिखकर save न करें।

> Live mode का Dhan REST code मैंने Dhan API v2 docs के हिसाब से लिखा है, पर अपने token से इसे test नहीं कर पाया। पहली बार चलाने पर output ज़रूर जाँच लें।
> अगर किसी option का securityId scrip master से न मिले, तो Dhan app में contract search करके उसका ID `CONTRACTS` में डाल दें, जैसे `CONTRACTS["NIFTY_22700_PE"] = 51341`।
>
> Not financial advice. एक दिन का profit यह साबित नहीं करता कि strategy आगे भी चलेगी।

In [ ]:
# ---------- Settings ----------
MODE = "replay"            # "replay" = 23 Oct 2026 का data (offline) | "live" = Dhan API से आज का data
TRADE_DATE = "2026-10-23"  # live mode में आज की date (YYYY-MM-DD)
EXPIRY = "2026-10-27"      # option expiry (live mode में बदलें)

UNDERLYINGS = {
    "NIFTY":     {"index_id": 13, "step": 50,  "lot": 65, "slippage": 0.5},
    "BANKNIFTY": {"index_id": 25, "step": 100, "lot": 30, "slippage": 1.0},
}
RULES = {
    "capital": 500000,
    "max_loss_per_trade": 5000,
    "daily_loss_limit": 15000,      # combined; इसके बाद कोई नई entry नहीं
    "max_trades_per_underlying": 2,
    "or_end": "09:30",              # opening range = 09:15-09:30
    "last_entry_bar": "14:55",      # trigger candle 15:00 तक close हो
    "flat_bar": "15:15",            # 15:15 वाली candle के close (15:20) पर बाहर
    "stop_pct": 30.0,               # premium से 30% नीचे stop
    "target_r": 2.0,                # 2R target
}

In [ ]:
# ---------- 23 Oct 2026 का saved data (5-min candles, IST bar start) ----------
# idx_* = index candles, बाकी = option candles. Dhan से लिया गया।
REPLAY_DATA = {
 "idx_NIFTY": [
  {
   "t": "09:15",
   "o": 22742.2,
   "h": 22787.25,
   "l": 22719.8,
   "c": 22726.25
  },
  {
   "t": "09:20",
   "o": 22728.65,
   "h": 22747.65,
   "l": 22699.7,
   "c": 22711.2
  },
  {
   "t": "09:25",
   "o": 22713.4,
   "h": 22763.5,
   "l": 22710.0,
   "c": 22726.45
  },
  {
   "t": "09:30",
   "o": 22729.85,
   "h": 22733.0,
   "l": 22676.0,
   "c": 22681.95
  },
  {
   "t": "09:35",
   "o": 22683.7,
   "h": 22695.2,
   "l": 22663.75,
   "c": 22686.9
  },
  {
   "t": "09:40",
   "o": 22690.4,
   "h": 22730.9,
   "l": 22688.75,
   "c": 22689.0
  },
  {
   "t": "09:45",
   "o": 22689.7,
   "h": 22703.15,
   "l": 22657.45,
   "c": 22661.7
  },
  {
   "t": "09:50",
   "o": 22661.7,
   "h": 22669.4,
   "l": 22630.05,
   "c": 22637.95
  },
  {
   "t": "09:55",
   "o": 22639.85,
   "h": 22649.95,
   "l": 22602.6,
   "c": 22609.15
  },
  {
   "t": "10:00",
   "o": 22609.0,
   "h": 22610.9,
   "l": 22554.9,
   "c": 22560.6
  },
  {
   "t": "10:05",
   "o": 22560.7,
   "h": 22583.4,
   "l": 22520.6,
   "c": 22523.05
  },
  {
   "t": "10:10",
   "o": 22526.25,
   "h": 22532.6,
   "l": 22492.0,
   "c": 22504.6
  }
 ],
 "idx_BANKNIFTY": [
  {
   "t": "09:15",
   "o": 54965.75,
   "h": 54990.5,
   "l": 54699.2,
   "c": 54757.0
  },
  {
   "t": "09:20",
   "o": 54753.1,
   "h": 54782.4,
   "l": 54647.15,
   "c": 54672.3
  },
  {
   "t": "09:25",
   "o": 54673.6,
   "h": 54815.3,
   "l": 54673.4,
   "c": 54767.4
  },
  {
   "t": "09:30",
   "o": 54770.35,
   "h": 54778.95,
   "l": 54691.05,
   "c": 54700.55
  },
  {
   "t": "09:35",
   "o": 54708.0,
   "h": 54738.2,
   "l": 54643.35,
   "c": 54679.2
  },
  {
   "t": "09:40",
   "o": 54685.1,
   "h": 54761.15,
   "l": 54658.35,
   "c": 54661.4
  },
  {
   "t": "09:45",
   "o": 54664.35,
   "h": 54687.3,
   "l": 54593.6,
   "c": 54607.7
  },
  {
   "t": "09:50",
   "o": 54612.8,
   "h": 54620.45,
   "l": 54521.25,
   "c": 54529.4
  },
  {
   "t": "09:55",
   "o": 54531.55,
   "h": 54564.55,
   "l": 54442.05,
   "c": 54442.05
  },
  {
   "t": "10:00",
   "o": 54444.05,
   "h": 54444.05,
   "l": 54305.1,
   "c": 54320.75
  },
  {
   "t": "10:05",
   "o": 54318.9,
   "h": 54391.25,
   "l": 54219.35,
   "c": 54233.4
  },
  {
   "t": "10:10",
   "o": 54239.9,
   "h": 54283.9,
   "l": 54175.2,
   "c": 54208.75
  }
 ],
 "NIFTY_22700_PE": [
  {
   "t": "09:30",
   "o": 118.55,
   "h": 143.55,
   "l": 117.7,
   "c": 141.3
  },
  {
   "t": "09:35",
   "o": 139.55,
   "h": 149.0,
   "l": 131.4,
   "c": 136.55
  },
  {
   "t": "09:40",
   "o": 135.65,
   "h": 139.1,
   "l": 116.0,
   "c": 138.25
  },
  {
   "t": "09:45",
   "o": 137.6,
   "h": 150.75,
   "l": 130.1,
   "c": 147.5
  },
  {
   "t": "09:50",
   "o": 147.3,
   "h": 164.0,
   "l": 142.7,
   "c": 160.5
  },
  {
   "t": "09:55",
   "o": 159.75,
   "h": 180.0,
   "l": 154.65,
   "c": 175.05
  },
  {
   "t": "10:00",
   "o": 176.55,
   "h": 204.8,
   "l": 175.5,
   "c": 201.1
  },
  {
   "t": "10:05",
   "o": 200.5,
   "h": 231.2,
   "l": 186.6,
   "c": 228.65
  },
  {
   "t": "10:10",
   "o": 226.85,
   "h": 245.05,
   "l": 219.7,
   "c": 233.7
  }
 ],
 "BANKNIFTY_54600_PE": [
  {
   "t": "09:45",
   "o": 303.3,
   "h": 329.75,
   "l": 290.85,
   "c": 321.85
  },
  {
   "t": "09:50",
   "o": 320.25,
   "h": 360.0,
   "l": 314.25,
   "c": 357.0
  },
  {
   "t": "09:55",
   "o": 357.2,
   "h": 409.3,
   "l": 338.55,
   "c": 404.35
  },
  {
   "t": "10:00",
   "o": 416.45,
   "h": 489.75,
   "l": 412.2,
   "c": 477.25
  },
  {
   "t": "10:05",
   "o": 478.8,
   "h": 549.0,
   "l": 439.45,
   "c": 538.6
  },
  {
   "t": "10:10",
   "o": 533.25,
   "h": 559.4,
   "l": 505.4,
   "c": 538.9
  }
 ]
}

# Dhan securityIds (NSE_FNO) of the contracts used on 23 Oct 2026
CONTRACTS = {
 "NIFTY_22700_PE": 51341,
 "BANKNIFTY_54600_PE": 49428
}

In [ ]:
# ---------- Engine: सारे bars शुरू से replay, fixed rules ----------
import math

def hm(s):
    h, m = s.split(":")
    return int(h) * 60 + int(m)

def tstr(mins):
    return "%02d:%02d" % (mins // 60, mins % 60)

def tick_down(x):
    return math.floor(round(x * 20, 6)) / 20.0

def tick_up(x):
    return math.ceil(round(x * 20, 6)) / 20.0

def charges(buy_turnover, sell_turnover):
    """Index option round trip के अनुमानित charges (रुपये)."""
    brokerage = 40.0
    stt = 0.0015 * sell_turnover
    exch = 0.0003503 * (buy_turnover + sell_turnover)
    sebi = 0.000001 * (buy_turnover + sell_turnover)
    stamp = 0.00003 * buy_turnover
    gst = 0.18 * (brokerage + exch + sebi)
    return round(brokerage + stt + exch + sebi + stamp + gst, 2)

def replay(data, ltps=None):
    """data: {key: [{"t","o","h","l","c"}, ...]}. Returns the day's report dict."""
    ltps = ltps or {}
    R = RULES
    bars_of = lambda key: {b["t"]: b for b in data.get(key, [])}
    st = {}
    for u, cfg in UNDERLYINGS.items():
        idx = bars_of("idx_" + u)
        s = {"cfg": cfg, "idx": idx, "events": [], "trades": [], "pos": None, "used": set(),
             "done": False, "pending": None, "status": None}
        orb = [b for t, b in idx.items() if hm(t) < hm(R["or_end"])]
        if len(orb) < 3:
            s["status"] = "opening range forming (%d of 3 bars)" % len(orb)
            s["done"], s["or"] = True, None
        else:
            s["or"] = {"high": max(b["h"] for b in orb), "low": min(b["l"] for b in orb)}
        st[u] = s

    realized = 0.0
    for m in range(hm(R["or_end"]), hm(R["flat_bar"]) + 5, 5):
        t = tstr(m)
        for u, s in st.items():
            if s["pending"] or (s["done"] and not s["pos"]):
                continue
            cfg, lot, slip = s["cfg"], s["cfg"]["lot"], s["cfg"]["slippage"]
            if s["pos"]:                                   # 1. open position संभालो
                p = s["pos"]
                ob = bars_of(p["key"]).get(t)
                if ob is None:
                    s["pending"] = "need %s bars from %s" % (p["key"], t)
                    continue
                exit_px, why = None, None
                if ob["o"] <= p["stop"]:
                    exit_px, why = ob["o"] - slip, "stop (gap)"
                elif ob["l"] <= p["stop"]:
                    exit_px, why = p["stop"] - slip, "stop"
                elif ob["o"] >= p["target"]:
                    exit_px, why = ob["o"], "target"
                elif ob["h"] >= p["target"]:
                    exit_px, why = p["target"], "target"
                elif m >= hm(R["flat_bar"]):
                    exit_px, why = ob["c"] - slip, "flat by 15:20"
                if exit_px is None:
                    p["last"], p["last_bar"] = ob["c"], t
                    continue
                exit_px = max(round(exit_px, 2), 0.05)
                qty = p["lots"] * lot
                gross = (exit_px - p["entry"]) * qty
                ch = charges(p["entry"] * qty, exit_px * qty)
                tr = dict(p, exit=exit_px, exit_bar=t, exit_reason=why,
                          gross=round(gross, 2), charges=ch, net=round(gross - ch, 2))
                tr.pop("last", None); tr.pop("last_bar", None)
                s["trades"].append(tr)
                realized += gross - ch
                s["events"].append("%s bar: EXIT %s x%d at %.2f (%s), net Rs %.0f" % (t, p["key"], qty, exit_px, why, gross - ch))
                s["pos"] = None
                if not why.startswith("stop") or len(s["trades"]) >= R["max_trades_per_underlying"]:
                    s["done"] = True
                continue
            if m > hm(R["last_entry_bar"]):                # 2. नई entry देखो
                s["done"] = True
                continue
            if realized <= -R["daily_loss_limit"]:
                s["events"].append("%s: daily loss limit hit, no new entries" % t)
                s["done"] = True
                continue
            ib = s["idx"].get(t)
            if ib is None:
                s["pending"] = "need idx_%s bars from %s" % (u, t)
                continue
            side = None
            if ib["c"] > s["or"]["high"] and "CE" not in s["used"]:
                side = "CE"
            elif ib["c"] < s["or"]["low"] and "PE" not in s["used"]:
                side = "PE"
            if not side:
                continue
            strike = int(round(ib["c"] / cfg["step"]) * cfg["step"])
            key = "%s_%d_%s" % (u, strike, side)
            ob = bars_of(key).get(t)
            if ob is None:
                s["pending"] = "need %s bars from %s" % (key, t)
                continue
            s["used"].add(side)
            entry = round(ob["c"] + slip, 2)
            stop = tick_down(entry * (1 - R["stop_pct"] / 100))
            risk = entry - stop
            target = tick_up(entry + R["target_r"] * risk)
            lots = int(R["max_loss_per_trade"] // (risk * lot))
            if lots < 1:
                s["events"].append("%s bar: %s BLOCKED: risk Rs %.0f a lot > max loss" % (t, key, risk * lot))
                continue
            s["pos"] = {"key": key, "entry_bar": t, "index_close": ib["c"], "entry": entry, "stop": stop,
                        "target": target, "lots": lots, "qty": lots * lot,
                        "max_loss": round(risk * lots * lot, 2), "premium_paid": round(entry * lots * lot, 2)}
            s["events"].append("%s bar: BUY %s x%d (%d lot) at %.2f | stop %.2f | target %.2f | max loss Rs %.0f"
                               % (t, key, lots * lot, lots, entry, stop, target, risk * lots * lot))

    out = {"underlyings": {}, "realized_net": round(realized, 2)}
    mtm_total = 0.0
    for u, s in st.items():
        o = {"opening_range": s["or"], "events": s["events"], "trades": s["trades"],
             "pending": s["pending"], "open_position": None}
        if s["pos"]:
            p = dict(s["pos"])
            p["mark"] = ltps.get(p["key"], p.get("last", p["entry"]))
            p["mtm"] = round((p["mark"] - p["entry"]) * p["qty"], 2)
            mtm_total += p["mtm"]
            o["open_position"], o["status"] = p, "in a trade"
        elif s["status"]:
            o["status"] = s["status"]
        elif s["done"]:
            o["status"] = "done for the day"
        else:
            o["status"] = "watching"
        out["underlyings"][u] = o
    out["open_mtm"] = round(mtm_total, 2)
    out["day_net_incl_mtm"] = round(realized + mtm_total, 2)
    return out

In [ ]:
# ---------- Live mode: Dhan API v2 (सिर्फ़ data पढ़ना, कोई order नहीं) ----------
import datetime as dt
from zoneinfo import ZoneInfo
import requests

IST = ZoneInfo("Asia/Kolkata")
DHAN = "https://api.dhan.co/v2"
SCRIP_MASTER = "https://images.dhan.co/api-data/api-scrip-master.csv"
_creds = {}

def dhan_headers():
    if not _creds:
        from getpass import getpass
        _creds["client-id"] = input("Dhan client-id: ").strip()
        _creds["access-token"] = getpass("Dhan access token (छुपा रहेगा): ").strip()
    return {"Content-Type": "application/json", "Accept": "application/json", **_creds}

def dhan_intraday(security_id, segment, instrument, date=TRADE_DATE):
    """पूरे दिन की 5-min candles; सिर्फ़ close हो चुकी candles लौटाता है."""
    body = {"securityId": str(security_id), "exchangeSegment": segment, "instrument": instrument,
            "interval": "5", "oi": False,
            "fromDate": date + " 09:00:00", "toDate": date + " 15:30:00"}
    r = requests.post(DHAN + "/charts/intraday", json=body, headers=dhan_headers(), timeout=30)
    r.raise_for_status()
    d = r.json()
    now = dt.datetime.now(IST)
    out = []
    for ts, o, h, l, c in zip(d["timestamp"], d["open"], d["high"], d["low"], d["close"]):
        start = dt.datetime.fromtimestamp(ts, IST)
        if start + dt.timedelta(minutes=5) <= now:           # forming candle छोड़ दो
            out.append({"t": start.strftime("%H:%M"), "o": o, "h": h, "l": l, "c": c})
    return out

def dhan_ltp(security_ids):
    r = requests.post(DHAN + "/marketfeed/ltp", json={"NSE_FNO": [int(i) for i in security_ids]},
                      headers=dhan_headers(), timeout=30)
    r.raise_for_status()
    feed = r.json().get("data", {}).get("NSE_FNO", {})
    return {str(k): v.get("last_price") for k, v in feed.items()}

_master = None
def find_option_id(underlying, strike, opt_type, expiry=EXPIRY):
    """Dhan scrip master से option का securityId (NIFTY को FINNIFTY/BANKNIFTY से अलग रखता है)."""
    global _master
    import pandas as pd
    if _master is None:
        _master = pd.read_csv(SCRIP_MASTER, low_memory=False)
    m = _master
    hit = m[(m["SEM_EXM_EXCH_ID"] == "NSE") & (m["SEM_INSTRUMENT_NAME"] == "OPTIDX")
            & (m["SEM_OPTION_TYPE"] == opt_type)
            & (m["SEM_STRIKE_PRICE"].astype(float) == float(strike))
            & (m["SEM_EXPIRY_DATE"].astype(str).str.startswith(expiry))
            & (m["SEM_TRADING_SYMBOL"].astype(str).str.startswith(underlying + "-"))]
    if len(hit) != 1:
        raise RuntimeError("%s %s %s %s: %d matches in scrip master" % (underlying, strike, opt_type, expiry, len(hit)))
    return int(hit.iloc[0]["SEM_SMST_SECURITY_ID"])

def run_live_once():
    """आज का data खींचो, replay चलाओ; नए signal पर उस option का data लाकर फिर चलाओ."""
    data = {"idx_" + u: dhan_intraday(c["index_id"], "IDX_I", "INDEX") for u, c in UNDERLYINGS.items()}
    for _ in range(6):
        rep = replay(data)
        needs = [o["pending"].split()[1] for o in rep["underlyings"].values()
                 if o["pending"] and not o["pending"].split()[1].startswith("idx_")]
        needs = [k for k in needs if k not in data]
        if not needs:
            break
        for key in needs:
            u, strike, opt = key.split("_")
            if key not in CONTRACTS:
                CONTRACTS[key] = find_option_id(u, int(strike), opt)
            data[key] = dhan_intraday(CONTRACTS[key], "NSE_FNO", "OPTIDX")
    open_keys = [o["open_position"]["key"] for o in rep["underlyings"].values() if o["open_position"]]
    if open_keys:
        prices = dhan_ltp([CONTRACTS[k] for k in open_keys])
        rep = replay(data, {k: prices.get(str(CONTRACTS[k])) for k in open_keys if prices.get(str(CONTRACTS[k]))})
    return rep

In [ ]:
# ---------- चलाओ और नतीजा दिखाओ ----------
import pandas as pd

def show(rep):
    for u, o in rep["underlyings"].items():
        orng = o["opening_range"]
        print("%-10s status: %s | opening range: %s" % (u, o["status"],
              "%.2f / %.2f" % (orng["high"], orng["low"]) if orng else "-"))
        for e in o["events"]:
            print("    " + e)
        if o["open_position"]:
            p = o["open_position"]
            print("    OPEN %s entry %.2f mark %.2f MTM Rs %.2f" % (p["key"], p["entry"], p["mark"], p["mtm"]))
    rows = [dict(underlying=u, **t) for u, o in rep["underlyings"].items() for t in o["trades"]]
    if rows:
        df = pd.DataFrame(rows)[["underlying", "key", "entry_bar", "entry", "exit_bar", "exit", "exit_reason",
                                 "qty", "premium_paid", "gross", "charges", "net"]]
        display(df)
        invested = df["premium_paid"].sum()
        print("Premium invested: Rs %.2f | Net P&L: Rs %.2f | on premium: %.1f%% | on capital Rs %d: %.2f%%"
              % (invested, rep["realized_net"], 100 * rep["realized_net"] / invested,
                 RULES["capital"], 100 * rep["realized_net"] / RULES["capital"]))
    print("Realized: Rs %.2f | Open MTM: Rs %.2f | Day net: Rs %.2f"
          % (rep["realized_net"], rep["open_mtm"], rep["day_net_incl_mtm"]))

report = replay(REPLAY_DATA) if MODE == "replay" else run_live_once()
show(report)

In [ ]:
# ---------- (Optional) Live mode में हर 5 मिनट चलाओ, चाहें तो email भी ----------
# यह cell सिर्फ़ तब चलाएँ जब MODE = "live" हो। Colab tab खुला रहना चाहिए। रोकने के लिए Stop दबाएँ।
SEND_EMAIL = False          # True करें तो हर 15 मिनट Gmail से update (Gmail App Password चाहिए)
EMAIL_EVERY_MIN = 15

_mail = {}   # Gmail login अलग रखा है, Dhan headers में नहीं जाता

def send_email(subject, text):
    import smtplib
    from email.mime.text import MIMEText
    from getpass import getpass
    if not _mail:
        _mail["email"] = input("आपका Gmail address: ").strip()
        _mail["app_password"] = getpass("Gmail App Password: ").strip()
    msg = MIMEText(text, "plain", "utf-8")
    msg["Subject"], msg["From"], msg["To"] = subject, _mail["email"], _mail["email"]
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as s:
        s.login(_mail["email"], _mail["app_password"])
        s.send_message(msg)

if MODE == "live":
    import time, io, contextlib
    last_mail = None
    while dt.datetime.now(IST).strftime("%H:%M") <= "15:25":
        rep = run_live_once()
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            show(rep)
        print(dt.datetime.now(IST).strftime("%H:%M:%S"), "\n" + buf.getvalue())
        now = dt.datetime.now(IST)
        if SEND_EMAIL and (last_mail is None or (now - last_mail).seconds >= EMAIL_EVERY_MIN * 60):
            send_email("Paper trading update %s" % now.strftime("%H:%M"), buf.getvalue())
            last_mail = now
        time.sleep(300 - now.second)  # अगली 5-min candle close के बाद
else:
    print("MODE = 'replay' है, live loop नहीं चलेगा।")

## 23 Oct 2026 का नतीजा (paper)

| Option | Entry | Exit | Qty | Premium लगाया | Charges | Net |
|---|---|---|---|---|---|---|
| NIFTY 27-Oct-2026 22700 PE | 9:35 पर ₹141.80 | 10:05–10:10 के बीच ₹226.90 (target) | 65 (1 lot) | ₹9,217.00 | ₹79.53 | **+₹5,451.97** |
| BANKNIFTY 27-Oct-2026 54600 PE | 9:50 पर ₹322.85 | 10:05–10:10 के बीच ₹516.65 (target) | 30 (1 lot) | ₹9,685.50 | ₹81.18 | **+₹5,732.82** |
| **कुल** | | | | **₹18,902.50** | ₹160.71 | **+₹11,184.79** |

लगाए गए premium पर +59.2%, ₹5,00,000 paper capital पर +2.24%। उस दिन बाज़ार opening range से सीधा नीचे गिरा, इसलिए breakout वाला तरीका काम कर गया। Range में ऊपर-नीचे होने वाले दिनों में यही rules लगातार stop-loss भी लगवाते हैं।